# 1) Forecasting basics — Exercises

**Goals:** write the metrics yourself, derive which lags a day-ahead forecast may use, make the weekly baseline aware of holidays, run a rolling-origin backtest, and blend two baselines with a weight chosen honestly.

The setup cell loads the lecture's data (`demand`, `holiday`, `train` = 2023–2024, `test` = 2025). Demand in MW.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# forecast_data.py lives in the module folder: search upwards so this works from exercise/ and exercise/solutions/
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "forecast_data.py").exists())))
from forecast_data import load_data

warnings.filterwarnings("ignore", category=FutureWarning)
df = load_data()
demand, holiday = df["demand"], df["holiday"]
train, test = demand[:"2024"], demand["2025"]

### Warm-ups

**1. The metrics**

Write `scores(forecast, actual, scale)` returning a dict with MAE, RMSE, bias, MAPE (in %) and MASE (MAE / scale), and `mase_scale(train, season=168)` — the in-sample MAE of the seasonal naive forecast on the training data.

In [ ]:
def scores(forecast, actual, scale: float) -> dict:
    """MAE, RMSE, bias, MAPE (%) and MASE of a forecast."""
    ...  # YOUR CODE HERE


def mase_scale(train: pd.Series, season: int = 168) -> float:
    """MAE of the seasonal naive forecast on the training data."""
    ...  # YOUR CODE HERE

In [ ]:
toy = scores([110, 90, 130], [100, 100, 100], scale=10.0)
assert np.isclose(toy["MAE"], 50 / 3) and np.isclose(toy["bias"], 10.0) and np.isclose(toy["MASE"], 5 / 3)
assert np.isclose(toy["RMSE"], np.sqrt((100 + 100 + 900) / 3)) and np.isclose(toy["MAPE_%"], 50 / 3)
assert np.isclose(mase_scale(train), 129.8, atol=0.1), "the lecture's 129.8 MW"

**2. Which lags are allowed?**

A forecast for hour h (0–23) of day D is issued at `issue_hour` on day D−1, when demand is known up to the hour before `issue_hour`. Write `daily_lag(target_hour, issue_hour)` returning the smallest whole number of days k such that "the same hour k days earlier" is known — as a lag in hours (24·k). Check the lecture's rule, then: what changes for an intraday forecast issued at 06:00 on the same day D (`same_day=True`)?

In [ ]:
def daily_lag(target_hour: int, issue_hour: int, same_day: bool = False) -> int:
    """Smallest lag (hours, a multiple of 24) to the same hour on an earlier day that is known at issue time."""
    ...  # YOUR CODE HERE

In [ ]:
assert [daily_lag(h, 12) for h in (0, 11, 12, 23)] == [24, 24, 48, 48], "the lecture's rule"
assert daily_lag(5, 6, same_day=True) == 24 and daily_lag(18, 6, same_day=True) == 24
assert daily_lag(10, 0) == 48, "issued at midnight: nothing of the day of issue is known yet"

### Core

**3. A holiday-aware weekly baseline**

"Same hour last week" fails on holidays. Write `holiday_aware(demand, holiday, index)`: for a target hour on a **holiday**, use the same hour of the most recent Sunday before it; on a normal day whose day one week earlier **was a holiday**, use two weeks earlier; otherwise one week. Score it against the plain weekly baseline on 2025 — overall and on holidays.

In [ ]:
def holiday_aware(demand: pd.Series, holiday: pd.Series, index: pd.DatetimeIndex) -> pd.Series:
    """Weekly naive forecast that treats holidays like Sundays and skips holidays as references."""
    ...  # YOUR CODE HERE

In [ ]:
plain = demand.shift(168).loc[test.index]
aware = holiday_aware(demand, holiday, test.index)
h = holiday.loc[test.index]
result = pd.DataFrame({
    "plain weekly": {"MAE all": (plain - test).abs().mean(), "MAE holidays": (plain - test)[h].abs().mean()},
    "holiday-aware": {"MAE all": (aware - test).abs().mean(), "MAE holidays": (aware - test)[h].abs().mean()}}).T
assert result.loc["holiday-aware", "MAE holidays"] < result.loc["plain weekly", "MAE holidays"]
assert result.loc["holiday-aware", "MAE all"] < result.loc["plain weekly", "MAE all"]
result.round(1)

**4. A rolling-origin backtest**

Climatology had a bias of −29 MW in 2025. One possible reason: it was estimated once, from 2023–2024, and never updated. Test that idea. Write `rolling_climatology(demand, test_index)`: for each month of 2025, re-estimate the hour-of-week averages from **all data before that month** and forecast the month. Report the monthly bias of the fixed and the rolling version.

In [ ]:
def rolling_climatology(demand: pd.Series, test_index: pd.DatetimeIndex) -> pd.Series:
    """Hour-of-week climatology re-estimated at the start of every month from all earlier data."""
    ...  # YOUR CODE HERE

In [ ]:
fixed_table = train.groupby([train.index.dayofweek, train.index.hour]).mean()
fixed = pd.Series(fixed_table.loc[list(zip(test.index.dayofweek, test.index.hour))].to_numpy(), index=test.index)
rolling = rolling_climatology(demand, test.index)
assert rolling["2025-01"].equals(fixed["2025-01"]), "January uses exactly the training data"
monthly = pd.DataFrame({"bias fixed": (fixed - test).groupby(test.index.month).mean(),
                        "bias rolling": (rolling - test).groupby(test.index.month).mean()}).rename_axis("month")
print(f"bias over 2025: fixed {(fixed - test).mean():.1f} MW, rolling {(rolling - test).mean():.1f} MW")
monthly.round(1).T

**5. Interpretation**

What did the holiday rule fix, and what could it not fix? Did re-estimating climatology every month remove the bias — and if not, why not? What does that tell you about baselines that average over long periods?

*Your answer here.*

### Challenge

**6. Blending two baselines — honestly**

A weighted blend w · (last week) + (1 − w) · (climatology) can beat both. Choose w from [0, 0.1, …, 1] **without looking at 2025**: estimate climatology on 2023 only, pick the w that minimises MAE on 2024, then refit climatology on 2023–2024 and evaluate that w on 2025. Compare with choosing w on 2025 itself.

In [ ]:
def choose_weight(demand: pd.Series, weights) -> tuple[float, pd.Series]:
    """Weight chosen on 2024 (climatology from 2023), and the MAE on 2024 for every weight."""
    ...  # YOUR CODE HERE

In [ ]:
weights = np.round(np.linspace(0, 1, 11), 1)
w_star, valid_mae = choose_weight(demand, weights)
weekly_test = demand.shift(168).loc[test.index]
test_mae = pd.Series({w: ((w * weekly_test + (1 - w) * fixed) - test).abs().mean() for w in weights})
assert 0 < w_star < 1, "a real blend, not one of the two baselines"
print(f"weight chosen on 2024: {w_star} -> 2025 MAE {test_mae[w_star]:.1f} MW "
      f"(last week alone {test_mae[1.0]:.1f}, climatology alone {test_mae[0.0]:.1f}; "
      f"best weight in hindsight {test_mae.idxmin()} -> {test_mae.min():.1f})")
pd.DataFrame({"MAE 2024 (choosing)": valid_mae, "MAE 2025 (testing)": test_mae}).round(1).T

**7. Interpretation**

Why does a blend of two mediocre baselines beat both? Why must the weight be chosen on 2024 and not on 2025 — and how much did the honest choice cost compared with hindsight?

*Your answer here.*